# 출처 검증형 deep research

제공한 공식 자료만 조회하는 도구를 가진 조사 에이전트를 실행하고 근거 파일을 남깁니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.

In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

한 가지 동작을 실행하고 출력으로 확인합니다.

In [3]:
# 검증된 제공 자료만 조회하는 조사 도구와 공유 백엔드를 준비합니다.
from dotenv import load_dotenv
from langchain.tools import tool
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
load_dotenv(Path('.env'))
provided = {
 'overview': 'Deep Agents is an agent harness built on LangChain and LangGraph. It provides filesystem, subagents, and context management.',
 'release': 'In 0.7.0 TodoListMiddleware became opt-in and the authored base prompt became empty. The 65% value was one gpt-5.5 hello-turn comparison.'
}
@tool
def source_lookup(source_id: str) -> str:
    """Return one pre-verified course source by its id: overview or release."""
    return provided.get(source_id, 'UNKNOWN SOURCE')
workspace = Path('labs/part6/ch27-deepagents/workspace').resolve()
backend = FilesystemBackend(root_dir=workspace)
researcher = {'name':'source-reviewer','description':'Verify claims against provided sources.',
 'system_prompt':'Use source_lookup. Return source id, supported claim, and limitation.', 'tools':[source_lookup]}
agent = create_deep_agent(model='openai:gpt-5.4-mini', tools=[source_lookup], subagents=[researcher], backend=backend)

## 단계 2

한 가지 동작을 실행하고 출력으로 확인합니다.

In [4]:
# 두 자료의 검증을 위임하고 근거·한계·출처 ID를 파일로 저장합니다.
prompt = '''Delegate overview and release checks to source-reviewer. Then write /research/final.md with
three headings: Verified claims, Scope limits, Source ids. Use only source_lookup evidence.'''
result = agent.invoke({'messages':[{'role':'user','content':prompt}]})
final = result['messages'][-1]
usage = getattr(final, 'usage_metadata', None)
print('최종 응답:', final.text[:500])
print('사용량:', usage)
saved = backend.read('/research/final.md')
print('저장 파일:', saved.file_data['content'][:700])
assert all(heading in saved.file_data['content'] for heading in ['Verified claims','Scope limits','Source ids'])

최종 응답: Done.
사용량: {'input_tokens': 2367, 'output_tokens': 6, 'total_tokens': 2373, 'input_token_details': {'cache_creation': 0, 'cache_read': 2048}, 'output_token_details': {'reasoning': 0}}
저장 파일: # Verified claims
- Deep Agents is an agent harness built on LangChain and LangGraph.
- It provides filesystem, subagents, and context management.
- In 0.7.0, TodoListMiddleware became opt-in and the authored base prompt became empty.
- The 65% value was one gpt-5.5 hello-turn comparison.

# Scope limits
- These checks are verified only from source_lookup evidence.
- The overview evidence is limited to the provided one-sentence description.
- The release evidence is limited to the single stated 0.7.0 note and comparison metric; no broader release context is available.

# Source ids
- overview
- release
